# Geodesic Distances via the Heat Method

Computing the geodesic distance from a source point in a weighted domain is classically done by the **Eikonal equation** $|\nabla d| = 1/\rho$, solved by fast marching. A more elegant approach is the **Cole–Hopf transformation**, which linearizes the Eikonal equation via a change of variables related to heat diffusion.

## Cole–Hopf transformation

The **Cole–Hopf substitution** $h = e^{-d/\varepsilon}$ transforms the weighted Eikonal equation
$$
\rho\,|\nabla d|^2 - \varepsilon\,\rho\,\Delta d = 1
$$
into the **linear elliptic equation**:
$$
\text{div}(\rho\,\nabla h) = -\frac{h}{\varepsilon}.
$$
With Dirichlet condition $h = 1$ at the source and $h \to 0$ far away, the solution gives $d = -\varepsilon \log h$. As $\varepsilon \to 0$, $d$ converges to the true geodesic distance (solution of the Eikonal).

## The heat method (Crane et al.)

A related approach by Crane, Weischedel and Wardetzky (2013) solves geodesics in two steps:
1. **Heat flow**: solve $\partial_t u = \Delta u$ for a short time $t$ starting from a point source. Normalize the gradient: $X = -\nabla u / |\nabla u|$.
2. **Poisson**: find $\phi$ such that $\Delta \phi = \text{div}(X)$. Then $\phi$ approximates the geodesic distance.

## Weighted geodesics

A **weight function** $\rho(x) > 0$ models varying terrain cost: $\rho$ large means moving through that region is cheap (fast), $\rho$ small means expensive (slow). This models **anisotropic diffusion** and **Riemannian metrics** on images.

## Environment

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatLogSlider, IntSlider

plt.rcParams['figure.dpi'] = 120

## Finite difference discretization

We discretize on an $n \times n$ grid with periodic or Neumann boundary conditions. The weighted Laplacian $L_{\rho} = \nabla^\top \text{diag}(\rho) \nabla$ is assembled as a sparse matrix.

In [ ]:
def build_laplacian_2d(n, rho=None, periodic=True):
    """Build weighted Laplacian on n x n grid."""
    N = n * n
    if rho is None:
        rho = np.ones((n, n))
    rho_flat = rho.flatten()

    # Finite difference gradient operators (periodic or Neumann)
    diag_m1 = -np.ones(n)
    diag_p1 = np.ones(n)

    if periodic:
        dx = sp.diags([diag_m1, diag_p1, [1.], [-1.]],
                      [-1, 1, -(n-1), n-1], shape=(n, n))
    else:
        dx = sp.diags([diag_m1[1:], diag_p1[:-1]], [-1, 1], shape=(n, n))

    Dx = sp.kron(sp.eye(n), dx)  # x-differences
    Dy = sp.kron(dx, sp.eye(n))  # y-differences
    Grad = sp.vstack([Dx, Dy])  # 2n^2 x n^2 gradient

    Rho = sp.diags(np.tile(rho_flat, 2))  # replicate for x and y
    Delta = (Grad.T @ Rho @ Grad) / n**2
    return Delta, Grad


def cole_hopf_distance(n, sources, rho=None, epsilon=0.05):
    """Geodesic distance via Cole-Hopf: solve (Delta + I/eps) h = b."""
    N = n * n
    Delta, _ = build_laplacian_2d(n, rho, periodic=False)

    # Source term: Dirac masses
    b = np.zeros(N)
    for (si, sj) in sources:
        b[si * n + sj] = 1.0

    # System: (Delta + I/eps) h = b/eps, with h=1 at sources
    A = Delta + sp.eye(N) / epsilon
    # Force h = 1 at source rows
    src_idx = [si * n + sj for (si, sj) in sources]
    A = A.tolil()
    A[src_idx, :] = 0
    A[src_idx, src_idx] = 1.0
    A = A.tocsr()

    rhs = np.zeros(N)
    rhs[src_idx] = 1.0

    h = spla.spsolve(A, rhs)
    h = np.maximum(h, 1e-14)
    D = -epsilon * np.log(h).reshape(n, n)
    return D


print('Heat distance solver ready.')

## Geodesic distances with uniform weight

For uniform $\rho = 1$, the geodesic distance from a point source is simply the Euclidean distance. As $\varepsilon \to 0$, the Cole–Hopf solution converges to this.

In [ ]:
n = 80
sources = [(n//2, n//2)]

eps_values = [2.0, 0.5, 0.1, 0.02]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

# Euclidean reference
i_idx, j_idx = np.mgrid[0:n, 0:n]
d_eucl = np.sqrt((i_idx - n//2)**2 + (j_idx - n//2)**2) / n

for ax, eps in zip(axes, eps_values):
    D = cole_hopf_distance(n, sources, epsilon=eps)
    D = D / D.max()
    im = ax.imshow(D, cmap='inferno', vmin=0, vmax=1)
    ax.contour(D, levels=10, colors='white', linewidths=0.7, alpha=0.7)
    ax.plot(n//2, n//2, 'c*', ms=10)
    ax.set_title(fr'$\varepsilon = {eps}$', fontsize=10)
    ax.axis('off')

fig.suptitle(r'Cole–Hopf distance: as $\varepsilon\to 0$ converges to Euclidean', y=1.02)
plt.tight_layout()
plt.show()

## Weighted geodesics: obstacle and terrain

We set $\rho$ to be very small inside an obstacle region (high cost = near-zero weight), forcing the geodesic to go around. We also show a smooth terrain where valleys are cheap and ridges are expensive.

In [ ]:
n = 80

# Obstacle: horizontal barrier with a gap
rho_obstacle = np.ones((n, n))
wall_row = slice(30, 50)
wall_col_l = slice(5, 35)
wall_col_r = slice(45, 75)
rho_obstacle[wall_row, wall_col_l] = 1e-4
rho_obstacle[wall_row, wall_col_r] = 1e-4

# Smooth terrain: ridges and valleys
ii, jj = np.mgrid[0:n, 0:n] / n
rho_terrain = 0.1 + 0.9 * (0.5 + 0.5 * np.cos(4 * np.pi * ii) * np.cos(3 * np.pi * jj))

sources_tl = [(10, 10)]
eps = 0.04

D_obs = cole_hopf_distance(n, sources_tl, rho=rho_obstacle, epsilon=eps)
D_terr = cole_hopf_distance(n, sources_tl, rho=rho_terrain, epsilon=eps)
D_eucl = cole_hopf_distance(n, sources_tl, rho=None, epsilon=eps)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (D, rho, title) in zip(axes, [
    (D_eucl, None, 'Uniform $\\rho=1$'),
    (D_obs, rho_obstacle, 'Wall obstacle'),
    (D_terr, rho_terrain, 'Smooth terrain'),
]):
    D_n = D / (D.max() + 1e-14)
    if rho is not None:
        ax.imshow(rho.T, cmap='gray', origin='lower', alpha=0.4)
    im = ax.imshow(D_n.T, cmap='plasma', origin='lower', alpha=0.7, vmin=0, vmax=1)
    ax.contour(D_n.T, levels=12, colors='white', linewidths=0.6, origin='lower', alpha=0.8)
    ax.plot(10, 10, 'c*', ms=12)
    ax.set_title(title, fontsize=10); ax.axis('off')

fig.suptitle('Weighted geodesic distances via Cole–Hopf', y=1.02)
plt.tight_layout()
plt.show()

## Multiple sources: Voronoi diagram

With multiple sources, the Cole–Hopf solution gives the distance to the **nearest source**. The Voronoi cells are the regions where each source is nearest.

In [ ]:
rng = np.random.default_rng(17)
n_src = 8
src_ij = (rng.integers(10, n-10, (n_src, 2))).tolist()

D_multi = cole_hopf_distance(n, src_ij, rho=None, epsilon=0.03)

# Voronoi: for each pixel, which source is nearest?
D_each = np.stack([
    cole_hopf_distance(n, [src], rho=None, epsilon=0.03)
    for src in src_ij
], axis=2)
voronoi = D_each.argmin(axis=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
D_n = D_multi / D_multi.max()
axes[0].imshow(D_n.T, cmap='viridis', origin='lower')
axes[0].contour(D_n.T, levels=12, colors='white', linewidths=0.6, origin='lower', alpha=0.7)
for (si, sj) in src_ij:
    axes[0].plot(si, sj, 'w*', ms=10)
axes[0].set_title('Distance to nearest source'); axes[0].axis('off')

axes[1].imshow(voronoi.T, cmap='tab10', origin='lower', vmin=0, vmax=9, alpha=0.8)
axes[1].contour(D_n.T, levels=12, colors='k', linewidths=0.5, origin='lower', alpha=0.4)
for (si, sj) in src_ij:
    axes[1].plot(si, sj, 'w*', ms=10)
axes[1].set_title('Voronoi cells (geodesic)'); axes[1].axis('off')

plt.tight_layout()
plt.show()

## Interactive: move source and adjust $\varepsilon$

Move the source point and change $\varepsilon$ to see how the Cole–Hopf approximation converges.

In [ ]:
def show_geodesic(src_x=40, src_y=40, log_eps=-1.5):
    eps = 10**log_eps
    D = cole_hopf_distance(n, [(src_x, src_y)], rho=rho_terrain, epsilon=eps)
    D_n = D / (D.max() + 1e-14)

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(rho_terrain.T, cmap='gray', origin='lower')
    axes[0].plot(src_x, src_y, 'r*', ms=12)
    axes[0].set_title('Terrain weight $\\rho$'); axes[0].axis('off')

    axes[1].imshow(D_n.T, cmap='plasma', origin='lower', vmin=0, vmax=1)
    axes[1].contour(D_n.T, levels=12, colors='white', linewidths=0.6, origin='lower', alpha=0.8)
    axes[1].plot(src_x, src_y, 'c*', ms=12)
    axes[1].set_title(fr'Geodesic distance ($\varepsilon={eps:.4f}$)')
    axes[1].axis('off')
    plt.tight_layout(); plt.show()

interact(show_geodesic,
         src_x=IntSlider(value=40, min=5, max=n-5, step=5, description='src $x$'),
         src_y=IntSlider(value=40, min=5, max=n-5, step=5, description='src $y$'),
         log_eps=FloatLogSlider(value=-1.5, min=-2.5, max=0.5, step=0.25,
                                description='$\\log_{10}\\varepsilon$'));

## Bibliographical resources

- Crane, K., Weischedel, C. and Wardetzky, M. (2013). Geodesics in heat: A transfer operator approach to distance, distance estimation and obstacle avoidance. *ACM Transactions on Graphics*, 32(5), Article 152.
- Hopf, E. (1950). The partial differential equation $u_t + u u_x = \mu u_{xx}$. *Communications on Pure and Applied Mathematics*, 3(3), 201–230.
- Varadhan, S. R. S. (1967). On the behavior of the fundamental solution of the heat equation with variable coefficients. *Communications on Pure and Applied Mathematics*, 20(2), 431–455.
- Sethian, J. A. (1999). *Level Set Methods and Fast Marching Methods* (2nd ed.). Cambridge University Press.
- Kimmel, R. and Sethian, J. A. (1998). Computing geodesic paths on manifolds. *Proceedings of the National Academy of Sciences*, 95(15), 8431–8435.